# Khmer History RAG - User-Side Chat Pipeline

The user-facing half of the project. Chunks are already built, the retriever is
already fine-tuned. This turns that into an interactive chat.

## Pipeline
```
User asks in Khmer (Gradio textbox)
   -> embed_query()    : khmer-retriever-finetuned (e5 fine-tuned, 384-dim)
                         text = "query: " + question
   -> hybrid search    : alpha*dense + (1-alpha)*BM25  ->  top 50 candidates
   -> rerank()         : BAAI/bge-reranker-v2-m3 reads each (question, passage)
                         pair together and reorders  ->  top 5
   -> generate()       : Gemini writes a grounded Khmer answer from those 5
   -> Gradio chat      : answer + history
```

## Why the reranker

Measured on the 59 held-out reworded test questions (the hard set, where the
question avoids the passage's own wording):

| System | Recall@1 | Recall@5 |
|---|---|---|
| fine-tuned e5 alone | 20 / 59 | 40 / 59 |
| hybrid (BM25 + e5) | 23 / 59 | 42 / 59 |
| **hybrid -> rerank top 50 -> top 5** | **37 / 59** | **53 / 59** |

Recall@5 goes from 0.712 to 0.898. The reranker is not fine-tuned and needs no
training data. It costs about 2 s per question on a GPU, much more on CPU.

## Passage text

Chunks are encoded as `historical_era_heading + ": " + text_payload`, which is
what the retriever was fine-tuned against and what every number above was
measured on. The language model is shown `text_payload` alone, so no section
heading leaks into the answer.

## 1. Imports & configuration

In [1]:
import os
import json
import numpy as np
from pathlib import Path

from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from khmernltk import word_tokenize
import gradio as gr

# ---- paths ----
# Works whether the notebook runs from the repo root or from src/.
ROOT = Path.cwd()
if not (ROOT / "data").is_dir() and (ROOT.parent / "data").is_dir():
    ROOT = ROOT.parent

STORE_PATH = ROOT / "data" / "rag_store.json"
MODEL_PATH = ROOT / "khmer-retriever-finetuned"

# ---- retrieval knobs ----
TOP_K  = 5       # chunks handed to the LLM
POOL   = 50      # candidates the reranker sees (measured: 50 matches 100, half the cost)
ALPHA  = 0.5     # weight on dense; (1-ALPHA) on BM25

# ---- reranker ----
USE_RERANKER   = True
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"

# ---- e5 prefixes (this model was fine-tuned with them; see its README) ----
QUERY_PREFIX   = "query: "
PASSAGE_PREFIX = "passage: "

# ---- Gemini ----
GEMINI_MODEL = "gemini-2.5-flash"

print("ROOT       =", ROOT)
print("store      =", STORE_PATH, "-> exists:", STORE_PATH.exists())
print("model dir  =", MODEL_PATH, "-> exists:", MODEL_PATH.exists())

/Users/macbookpro/Desktop/nlp6/Khmer_History_retrieval_argumented_generation/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1106 15:36:05.211000 15856 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


ROOT       = /Users/macbookpro/Desktop/nlp6/Khmer_History_retrieval_argumented_generation
store      = /Users/macbookpro/Desktop/nlp6/Khmer_History_retrieval_argumented_generation/data/rag_store.json -> exists: True
model dir  = /Users/macbookpro/Desktop/nlp6/Khmer_History_retrieval_argumented_generation/khmer-retriever-finetuned -> exists: True


## 2. Load the Gemini API key (securely)

The key is read from the `GEMINI_API_KEY` environment variable, loaded from a
git-ignored `.env` file. **Never hardcode the key in this notebook.**

Create a file named `.env` in the project root containing one line:
```
GEMINI_API_KEY=your_real_key_here
```
(See `.env.example`.) If you skip this, retrieval still works but generation is
disabled and the app falls back to showing the raw retrieved chunks.

In [2]:
from dotenv import load_dotenv

load_dotenv(ROOT / ".env")   # loads GEMINI_API_KEY into the environment

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

gemini_client = None
if GEMINI_API_KEY:
    from google import genai
    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    print("Gemini client ready (key loaded from environment).")
else:
    print("WARNING: GEMINI_API_KEY not found. Generation disabled -> will show raw chunks.")
    print("Create a .env file with GEMINI_API_KEY=... to enable Gemini answers.")

Gemini client ready (key loaded from environment).


## 3. Load the store and the retriever model

In [3]:
with open(STORE_PATH, encoding="utf-8") as f:
    store = json.load(f)

chunks    = store["chunks"]                          # 908 chunk dicts
metas     = [c["metadata"] for c in chunks]
chunk_ids = [c["chunk_id"] for c in chunks]

# Shown to the language model: the clean passage, no heading.
texts = [c["text_payload"] for c in chunks]

# Indexed for retrieval: heading + passage. This is what the retriever was
# fine-tuned against, and what the reported scores were measured on.
index_texts = [m["historical_era_heading"] + ": " + t for m, t in zip(metas, texts)]

print(f"Loaded {len(chunks)} chunks.")

weights = MODEL_PATH / "model.safetensors"
if not weights.exists():
    raise FileNotFoundError(
        f"{weights} is missing.\n"
        f"model.safetensors is git-ignored because it is 470 MB, over GitHub's limit.\n"
        f"Copy it from the shared Drive folder (khmer_rag/khmer-retriever-finetuned/) "
        f"into {MODEL_PATH}/, or set MODEL_PATH to the Hugging Face repo id."
    )

model = SentenceTransformer(str(MODEL_PATH))
print("Retriever loaded:", MODEL_PATH.name)

Loaded 908 chunks.


Loading weights: 0it [00:00, ?it/s]
[transformers] BertModel LOAD REPORT from: /Users/macbookpro/Desktop/nlp6/Khmer_History_retrieval_argumented_generation/khmer-retriever-finetuned
Key                                                                       | Status     | 
--------------------------------------------------------------------------+------------+-
encoder.block.{0, 1, 2, 3, 4, 5, 6, 7}.layer.1.DenseReluDense.wi_0.weight | UNEXPECTED | 
decoder.block.{0, 1, 2, 3, 4, 5, 6, 7}.layer.0.SelfAttention.v.weight     | UNEXPECTED | 
decoder.block.{0, 1, 2, 3, 4, 5, 6, 7}.layer.0.SelfAttention.k.weight     | UNEXPECTED | 
decoder.block.{0, 1, 2, 3, 4, 5, 6, 7}.layer.1.EncDecAttention.k.weight   | UNEXPECTED | 
decoder.block.{0, 1, 2, 3, 4, 5, 6, 7}.layer.0.SelfAttention.q.weight     | UNEXPECTED | 
encoder.block.{0, 1, 2, 3, 4, 5, 6, 7}.layer.0.SelfAttention.q.weight     | UNEXPECTED | 
encoder.block.{0, 1, 2, 3, 4, 5, 6, 7}.layer.1.DenseReluDense.wo.weight   | UNEXPECTED | 
decoder.

Retriever loaded: khmer-retriever-finetuned


## 4. Re-encode all chunks (dense index)

We encode every chunk now, with the loaded model, so the chunk vectors and the
query vectors come from the identical encode path. `normalize_embeddings=True`
means a dot product equals cosine similarity. Runs in a few seconds on GPU.

In [4]:
chunk_emb = model.encode(
    [PASSAGE_PREFIX + t for t in index_texts],
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True,
)
chunk_emb = np.asarray(chunk_emb, dtype=np.float32)
print("Dense index shape:", chunk_emb.shape)   # (908, 384)

Batches: 100%|██████████| 29/29 [00:12<00:00,  2.34it/s]


Dense index shape: (908, 384)


## 5. Build the BM25 (lexical) index

BM25 uses **no embeddings** — it matches overlapping words. We tokenize each
chunk with the Khmer tokenizer so word boundaries are correct (Khmer has no
spaces between words).

In [5]:
tokenized_chunks = [word_tokenize(t) for t in index_texts]
bm25 = BM25Okapi(tokenized_chunks)
print("BM25 index built over", len(tokenized_chunks), "chunks.")

| 2026-11-06 15:36:23,861 | INFO | khmer-nltk | Loaded model from /Users/macbookpro/Desktop/nlp6/Khmer_History_retrieval_argumented_generation/.venv/lib/python3.14/site-packages/khmernltk/word_tokenize/sklearn_crf_ner_10000.sav |


BM25 index built over 908 chunks.


## 5b. Load the reranker

A cross-encoder reads the question and one passage **together** and scores the
pair, instead of comparing two separately produced vectors. Far more accurate,
far slower, so it only ever sees the shortlist of `POOL` candidates.

Downloads about 2.3 GB on first use. Needs no fine-tuning and no training data.
Set `USE_RERANKER = False` above to skip it: retrieval still works, Recall@5
falls from 0.898 to about 0.712.

In [7]:
reranker = None
if USE_RERANKER:
    try:
        from FlagEmbedding import FlagReranker
    except ImportError:
        raise ImportError(
            "FlagEmbedding is not installed.  pip install FlagEmbedding\n"
            "Or set USE_RERANKER = False to run without reranking."
        )
    import torch
    on_gpu = torch.cuda.is_available()
    reranker = FlagReranker(RERANKER_MODEL, use_fp16=on_gpu)
    print(f"Reranker loaded: {RERANKER_MODEL}  (GPU: {on_gpu})")
    if not on_gpu:
        print("WARNING: no GPU. Reranking will take several seconds per question.")
else:
    print("Reranker disabled. Recall@5 about 0.712 instead of 0.898.")

KeyboardInterrupt: 

## 6. Retrieval functions (embed query -> search -> rank)

In [ ]:
def embed_query(question: str) -> np.ndarray:
    """Embed the question with the same model as the chunks."""
    return model.encode(QUERY_PREFIX + question, normalize_embeddings=True).astype(np.float32)


def dense_search(question: str) -> np.ndarray:
    """Semantic search: cosine similarity against every chunk."""
    return chunk_emb @ embed_query(question)


def bm25_search(question: str) -> np.ndarray:
    """Lexical search: keyword overlap, no embeddings."""
    return np.asarray(bm25.get_scores(word_tokenize(question)), dtype=np.float32)


def _minmax(x: np.ndarray) -> np.ndarray:
    return (x - x.min()) / (x.max() - x.min() + 1e-9)


def retrieve(question: str, top_k: int = TOP_K, pool: int = POOL, alpha: float = ALPHA):
    """Hybrid shortlist, then rerank, then take top_k.

    Returns dicts with chunk_id, text, page, url, score, and rerank_score.
    `score` is the hybrid score; `rerank_score` is the cross-encoder's, which
    is an absolute relevance estimate and is what the relevance gate uses.
    """
    hybrid = alpha * _minmax(dense_search(question)) + (1 - alpha) * _minmax(bm25_search(question))

    if reranker is None:
        idx = np.argsort(hybrid)[::-1][:top_k]
        return [_hit(i, float(hybrid[i]), None) for i in idx]

    # Shortlist first; the cross-encoder is too slow to run over all 908.
    cand = np.argsort(hybrid)[::-1][:max(pool, top_k)]

    # The reranker takes the raw question. "query: " is an e5 convention and
    # would only confuse it.
    pairs = [[question, index_texts[i]] for i in cand]
    rr = reranker.compute_score(pairs, normalize=True)
    if not isinstance(rr, list):
        rr = [rr]

    best = sorted(zip(rr, cand), key=lambda x: -x[0])[:top_k]
    return [_hit(i, float(hybrid[i]), float(s)) for s, i in best]


def _hit(i, hybrid_score, rerank_score):
    return {
        "chunk_id":     chunk_ids[i],
        "text":         texts[i],          # clean passage for the LLM
        "page":         metas[i].get("page_title", ""),
        "url":          metas[i].get("source_url", ""),
        "score":        hybrid_score,
        "rerank_score": rerank_score,
    }

## 7. Generation (Gemini writes the grounded Khmer answer)

We stuff the retrieved chunks into a prompt and instruct Gemini to answer **only**
from that context, in Khmer. If no API key is set, we fall back to returning the
top chunks directly so the pipeline still demonstrates retrieval.

In [ ]:
SYSTEM_INSTRUCTION = (
    "You are a helpful assistant that answers questions about Khmer (Cambodian) "
    "history. Answer ONLY using the provided context passages. If the answer is "
    "not in the context, say in Khmer that you do not have enough information "
    "(\u179f\u17bc\u1798\u1791\u17c4\u179f \u1781\u17d2\u1789\u17bb\u17c6\u1798\u17b7\u1793\u1798\u17b6\u1793\u1796\u17d0\u178f\u17cc\u1798\u17b6\u1793\u1782\u17d2\u179a\u1794\u17cb\u1782\u17d2\u179a\u17b6\u1793\u17cb). "
    "Always answer in Khmer language. The max words no more than 120 words"
)


def build_context(hits):
    """Format retrieved chunks into a numbered context block."""
    blocks = []
    for n, h in enumerate(hits, 1):
        blocks.append(f"[\u17a1{n}] ({h['page']})\n{h['text']}")
    return "\n\n".join(blocks)


def generate(question: str, hits) -> str:
    """Produce a grounded Khmer answer. Falls back to raw chunks if no LLM."""
    context = build_context(hits)

    if gemini_client is None:
        # No API key -> show retrieval result directly.
        return "⚠️ (Gemini off — រកឃើញឯកសារពាក់ព័ន្ធ):\n\n" + context

    prompt = (
        f"កម្មវត្ថុ (Context):\n{context}\n\n"
        f"សំណួរ (Question):\n{question}\n\n"
        f"សូមឆ្លើយជាភាសាខ្មែរ ដោយផ្អែកលើព័ត៌មានខាងលើប៉ុណ្ណោះ។"
    )

    resp = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config={"system_instruction": SYSTEM_INSTRUCTION, "temperature": 0.2},
    )
    # return only the grounded answer (no sources block)
    return resp.text.strip()


# ---- relevance gate -------------------------------------------------------
# Stops greetings and off-topic questions from reaching the LLM.
#
# With the reranker on we gate on its score, which is an absolute relevance
# estimate from 0 to 1 and far more meaningful than the hybrid score (which is
# min-max normalised, so it says more about dense/BM25 agreement than about
# relevance). Calibrate RERANK_THRESHOLD by printing rerank_score for a few
# real history questions and a few greetings, then pick a value between them.
RERANK_THRESHOLD    = 0.30   # used when the reranker is on
RELEVANCE_THRESHOLD = 0.60   # used when it is off

NO_INFO_MSG = "\u179f\u17bc\u1798\u1791\u17c4\u179f \u1781\u17d2\u1789\u17bb\u17c6\u1798\u17b7\u1793\u1798\u17b6\u1793\u1796\u17d0\u178f\u17cc\u1798\u17b6\u1793\u1782\u17d2\u179a\u1794\u17cb\u1782\u17d2\u179a\u17b6\u1793\u17cb\u178a\u17be\u1798\u17d2\u1794\u17b8\u1786\u17d2\u179b\u17be\u1799\u179f\u17c6\u178e\u17bd\u179a\u1793\u17c1\u17c7\u1791\u17c1\u17d4 \u179f\u17bc\u1798\u179f\u17bd\u179a\u17a2\u17c6\u1796\u17b8\u1794\u17d2\u179a\u179c\u178f\u17d2\u178f\u17b7\u179f\u17b6\u179f\u17d2\u178f\u17d2\u179a\u1780\u1798\u17d2\u1796\u17bb\u1787\u17b6\u17d4"


def answer_question(question: str) -> str:
    """Full RAG: retrieve -> rerank -> relevance gate -> generate."""
    question = (question or "").strip()
    if not question:
        return NO_INFO_MSG

    hits = retrieve(question)
    if not hits:
        return NO_INFO_MSG

    if hits[0]["rerank_score"] is not None:
        relevant = hits[0]["rerank_score"] >= RERANK_THRESHOLD
    else:
        relevant = hits[0]["score"] >= RELEVANCE_THRESHOLD

    if not relevant:
        return NO_INFO_MSG

    return generate(question, hits)

## 8. Quick test (before launching the UI)

In [ ]:
for q in [
    "\u178f\u17be\u17a2\u1784\u17d2\u1782\u179a\u179c\u178f\u17d2\u178f\u178f\u17d2\u179a\u17bc\u179c\u1794\u17b6\u1793\u179f\u17b6\u1784\u179f\u1784\u17cb\u1793\u17c5\u179f\u178f\u179c\u178f\u17d2\u179f\u1791\u17b8\u1794\u17c9\u17bb\u1793\u17d2\u1798\u17b6\u1793?",
    "hello",
]:
    print("Q:", q)
    hits = retrieve(q)
    print("   top chunk :", hits[0]["chunk_id"], "|", hits[0]["page"])
    print("   hybrid    : %.3f" % hits[0]["score"])
    if hits[0]["rerank_score"] is not None:
        print("   rerank    : %.3f" % hits[0]["rerank_score"])
    print("   answer    :", answer_question(q)[:200].replace("\n", " "))
    print()

## 9. Gradio chat interface

`fn` receives the latest message and the chat history. We ignore history for
retrieval (each question is retrieved independently) but Gradio still shows the
running conversation.

In [ ]:

def chat_fn(message, history):
    return answer_question(message)


# ---- load the Kh-Menghorn Khmer font and embed it as CSS ----
import base64

FONT_PATH = ROOT / "fonts" / "Kh-Menghorn.ttf"   # put the .ttf here
FONT_FAMILY = "Kh-Menghorn"

def build_font_css():
    if not FONT_PATH.exists():
        print(f"NOTE: font not found at {FONT_PATH} -> using default Khmer font.")
        print("      Download Kh-Menghorn and place the .ttf there to enable it.")
        return ""
    b64 = base64.b64encode(FONT_PATH.read_bytes()).decode()
    print(f"Font loaded: {FONT_PATH.name} ({len(b64)//1024} KB base64)")
    return (
        "@font-face {\n"
        f"  font-family: '{FONT_FAMILY}';\n"
        f"  src: url(data:font/ttf;base64,{b64}) format('truetype');\n"
        "  font-display: swap;\n"
        "}\n"
        "*, .gradio-container, .prose, textarea, input, button, .message, .bubble {\n"
        f"  font-family: '{FONT_FAMILY}', 'Noto Sans Khmer', 'Khmer OS', sans-serif !important;\n"
        "}\n"
    )

FONT_CSS = build_font_css()


demo = gr.ChatInterface(
    fn=chat_fn,
    title="ប្រវត្តិសាស្ត្រខ្មែរ RAG — Khmer History Chatbot",
    description="សួរសំណួរជាភាសាខ្មែរអំពីប្រវត្តិសាស្ត្រកម្ពុជា។ (Ask in Khmer about Cambodian history.)",
    examples=[
        "តើអង្គរវត្តត្រូវបានសាងសង់នៅសតវត្សទីប៉ុន្មាន?",
        "នគរហ្វូណនគឺអ្វី?",
        "ខ្មែរក្រហមគឺអ្វី?",
    ],
)

demo.launch(inbrowser=True, css=FONT_CSS)   # add share=True for a public link

NOTE: font not found at d:\School\AUPP\2026\Fall\NLP\Khmer_History_retrieval_argumented_generation\fonts\Kh-Menghorn.ttf -> using default Khmer font.
      Download Kh-Menghorn and place the .ttf there to enable it.
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "d:\School\AUPP\2026\Fall\NLP\Khmer_History_retrieval_argumented_generation\.venv\lib\site-packages\gradio\queueing.py", line 870, in process_events
    response = await route_utils.call_process_api(
  File "d:\School\AUPP\2026\Fall\NLP\Khmer_History_retrieval_argumented_generation\.venv\lib\site-packages\gradio\route_utils.py", line 418, in call_process_api
    output = await app.get_blocks().process_api(
  File "d:\School\AUPP\2026\Fall\NLP\Khmer_History_retrieval_argumented_generation\.venv\lib\site-packages\gradio\blocks.py", line 2698, in process_api
    result = await self.call_function(
  File "d:\School\AUPP\2026\Fall\NLP\Khmer_History_retrieval_argumented_generation\.venv\lib\site-packages\gradio\blocks.py", line 1961, in call_function
    prediction = await fn(*processed_input)
  File "d:\School\AUPP\2026\Fall\NLP\Khmer_History_retrieval_argumented_generation\.venv\lib\site-packages\gradio\utils.py", line 1083, in async_wrapper
    re